<a href="https://colab.research.google.com/github/Rokiafaysal/advanced-rag-chatbot/blob/main/rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf faiss-cpu sentence-transformers groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 16.0 MB/s eta 0:00:00


In [59]:
!pip install -U FlagEmbedding

In [18]:
!pip install transformers==4.44.0 FlagEmbedding --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.8/177.8 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.1/147.1 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 83.1 MB/s eta 0:00:00


In [1]:
import pypdf
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq

In [2]:
from FlagEmbedding import FlagReranker

In [3]:
from google.colab import userdata
client = Groq(api_key=userdata.get('Groq_key'))


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from pypdf import PdfReader
reader= PdfReader("/content/drive/MyDrive/front_matter.pdf")

In [6]:
text = "\n".join(p.extract_text() for p in reader.pages)


In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
def split_text(text, chunk_size=500,overlap=50):
  words = text.split()
  chunks=[]
  for i in range(0,len(words),chunk_size - overlap):
    chunk=" ".join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks




In [9]:

chunks = split_text(text)
print(f" num of chunks: {len(chunks)}")

 num of chunks: 52


In [10]:
embedding=model.encode(chunks)
index=faiss.IndexFlatL2(embedding.shape[1])
index.add(embedding)
print({index.ntotal})

{52}


In [11]:
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

In [20]:
def search_reranked_verbose(query, k=10, top_n=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    candidates = [chunks[i] for i in indices[0]]

    pairs = [[query, chunk] for chunk in candidates]
    scores = reranker.compute_score(pairs)

    ranked = sorted(zip(scores, candidates), reverse=True)

    for i, chunk in enumerate(candidates):
        print(f"{i+1}. {chunk[:100]}...")
        print("---")

    top = [chunk for _, chunk in ranked[:top_n]]
    for i, chunk in enumerate(top):
        print(f"{i+1}. {chunk[:100]}...")
        print("---")

    return top

results = search_reranked_verbose("what is deep learning")

1. D. J., Koh, P. W., and Ng, A. (2010). Tiled convolutional neural networks. In J. Laﬀerty, C. K. I. W...
---
2. G. (2012a). Acoustic modeling using deep belief networks. IEEE Trans. on Audio, Speech and Language ...
---
3. 446 Chen, T., Li, M., Li, Y., Lin, M., Wang, N., Wang, M., Xiao, T., Xu, B., Zhang, C., and Zhang, Z...
---
4. Neural Networks With Dropout. Master’s thesis, U. Toronto. 533 Srivastava, N. and Salakhutdinov, R. ...
---
5. 494 Coates, A., Lee, H., and Ng, A. Y. (2011). An analysis of single-layer networks in unsupervised ...
---
6. Alignments. InProceedings of ACL. 470 Krause, O., Fischer, A., Glasmachers, T., and Igel, C. (2013)....
---
7. H. C. and Mitchison, G. (1983). The function of dream sleep.Nature, 304, 111–114. 607 Cybenko, G.(19...
---
8. networks. InICLR’2014. 191 Goodfellow, I. J., Shlens, J., and Szegedy, C. (2014b). Explaining and ha...
---
9. Image Statistics: A probabilistic approach to early computational vision. Springer-Verlag. 364 Iba, ...
---
1

In [16]:
model_chat="llama-3.3-70b-versatile"

In [17]:
def call_llm(query, chunks, model_chat=model_chat, temperature=0.4, max_tokens=300):
  context = "\n".join(chunks)
  prompt = f"""
You are an assistant helping students understand this book.

context from this book
{context}
    Question: {query}
   answer based only on the context above.

  """

  response = client.chat.completions.create(
    model=model_chat,
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ],
    temperature=temperature,
    max_tokens=max_tokens
  )

  return response.choices[0].message.content



In [22]:
results = search_reranked_verbose("what is deep learning")
answer = call_llm("what is deep learning", results)
print(answer)

1. D. J., Koh, P. W., and Ng, A. (2010). Tiled convolutional neural networks. In J. Laﬀerty, C. K. I. W...
---
2. G. (2012a). Acoustic modeling using deep belief networks. IEEE Trans. on Audio, Speech and Language ...
---
3. 446 Chen, T., Li, M., Li, Y., Lin, M., Wang, N., Wang, M., Xiao, T., Xu, B., Zhang, C., and Zhang, Z...
---
4. Neural Networks With Dropout. Master’s thesis, U. Toronto. 533 Srivastava, N. and Salakhutdinov, R. ...
---
5. 494 Coates, A., Lee, H., and Ng, A. Y. (2011). An analysis of single-layer networks in unsupervised ...
---
6. Alignments. InProceedings of ACL. 470 Krause, O., Fischer, A., Glasmachers, T., and Igel, C. (2013)....
---
7. H. C. and Mitchison, G. (1983). The function of dream sleep.Nature, 304, 111–114. 607 Cybenko, G.(19...
---
8. networks. InICLR’2014. 191 Goodfellow, I. J., Shlens, J., and Szegedy, C. (2014b). Explaining and ha...
---
9. Image Statistics: A probabilistic approach to early computational vision. Springer-Verlag. 364 Iba, ...
---
1